In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import hamming_loss, precision_score, recall_score, f1_score
from PIL import Image


# CONFIG
test_dir = ""
int8_model_path = ""

batch_size = 16
threshold = 0.5

# INT8 MODELS → CPU ONLY
device = torch.device("cpu")

class_names = ["left", "right", "forward"]
num_classes = len(class_names)

# DATASET
class MultiLabelDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []

        for subfolder in sorted(os.listdir(root_dir)):
            sub_path = os.path.join(root_dir, subfolder)
            if not os.path.isdir(sub_path):
                continue

            # folder name: e.g. 1_0_1
            label_vec = [int(x) for x in subfolder.split("_")]

            for fname in os.listdir(sub_path):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.images.append(os.path.join(sub_path, fname))
                    self.labels.append(label_vec)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        # Load grayscale
        img = Image.open(img_path).convert("L")

        if self.transform:
            img = self.transform(img)

        # Expand 1 → 3 channels (ResNet expects 3)
        img = img.repeat(3, 1, 1)

        return img, label



# TRANSFORMS  (MATCH TRAINING!)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])

test_dataset = MultiLabelDataset(test_dir, transform=transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=False
)


# LOAD FULL INT8 MODEL (OPTION A)
print("Loading INT8 model...")
model = torch.jit.load("", map_location="cpu")
model.eval()
model.to(device)

# Sanity checks
print("Model type:", type(model))
print("First layer:", model.conv1)
#print("Model device:", next(model.parameters()).device)


# EVALUATION
def evaluate_model(model, loader, threshold=0.5):
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            probs = torch.sigmoid(outputs)
            preds = (probs >= threshold).int()

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    print("\n========== INT8 Evaluation ==========")
    print(f"Hamming Loss : {hamming_loss(all_labels, all_preds):.4f}")
    print(f"Precision    : {precision_score(all_labels, all_preds, average='micro', zero_division=0):.4f}")
    print(f"Recall       : {recall_score(all_labels, all_preds, average='micro', zero_division=0):.4f}")
    print(f"F1 Score     : {f1_score(all_labels, all_preds, average='micro', zero_division=0):.4f}")
    print("====================================")


# RUN
evaluate_model(model, test_loader)


Loading INT8 model...
Model type: <class 'torch.jit._script.RecursiveScriptModule'>
First layer: RecursiveScriptModule(original_name=ConvReLU2d)

========== INT8 Evaluation ==========
Hamming Loss : 0.2533
Precision    : 0.6633
Recall       : 0.9362
F1 Score     : 0.7765


In [13]:
print(model)

RecursiveScriptModule(
  original_name=GraphModule
  (conv1): RecursiveScriptModule(original_name=ConvReLU2d)
  (maxpool): RecursiveScriptModule(original_name=MaxPool2d)
  (layer1): RecursiveScriptModule(
    original_name=Module
    (0): RecursiveScriptModule(
      original_name=Module
      (conv1): RecursiveScriptModule(original_name=ConvReLU2d)
      (conv2): RecursiveScriptModule(original_name=ConvReLU2d)
      (conv3): RecursiveScriptModule(original_name=Conv2d)
      (downsample): RecursiveScriptModule(
        original_name=Module
        (0): RecursiveScriptModule(original_name=Conv2d)
      )
    )
    (1): RecursiveScriptModule(
      original_name=Module
      (conv1): RecursiveScriptModule(original_name=ConvReLU2d)
      (conv2): RecursiveScriptModule(original_name=ConvReLU2d)
      (conv3): RecursiveScriptModule(original_name=Conv2d)
    )
    (2): RecursiveScriptModule(
      original_name=Module
      (conv1): RecursiveScriptModule(original_name=ConvReLU2d)
      (conv

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from torch.ao.quantization import get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
from PIL import Image


# Configuration
DATA_ROOT = ""
FP32_MODEL_PATH = ""
INT8_MODEL_PATH = "resnet50_syn_int8_2.pth"

BATCH_SIZE = 16
CALIBRATION_BATCHES = 30

torch.backends.quantized.engine = "fbgemm"

# Dataset 
class MultiLabelDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for sub in os.listdir(root_dir):
            sub_path = os.path.join(root_dir, sub)
            if not os.path.isdir(sub_path):
                continue
            label = torch.tensor([int(x) for x in sub.split("_")], dtype=torch.float)
            for f in os.listdir(sub_path):
                if f.lower().endswith((".jpg", ".png", ".jpeg")):
                    self.samples.append((os.path.join(sub_path, f), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("L")
        img = self.transform(img)
        img = img.repeat(3, 1, 1)
        return img, label


# Calibration Transform
calib_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


# Load FP32 Model
def load_fp32_model(path):
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 3)
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()
    return model


# Main FX Quantization
def main():
    print("Load FP32 model")
    model = load_fp32_model(FP32_MODEL_PATH)
    print("FP32 MOdel ", model)

    print("Create example input...")
    example_input = torch.randn(1, 3, 224, 224)

    print("Setting qconfig mapping...")
    qconfig_mapping = get_default_qconfig_mapping("fbgemm")

    print("Preparing FX model...")
    model_prepared = prepare_fx(
        model,
        qconfig_mapping,
        example_input
    )

    print("Loading calibration data...")
    calib_dataset = MultiLabelDataset(DATA_ROOT, calib_transform)
    calib_loader = DataLoader(calib_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print("Running calibration...")
    with torch.no_grad():
        for i, (x, _) in enumerate(calib_loader):
            model_prepared(x)
            if i >= CALIBRATION_BATCHES:
                break

    print("Converting to INT8...")
    model_int8 = convert_fx(model_prepared)
    print("After converting int8 model ",model_int8)

    print("Saving INT8 model...")
    # model_int8.eval()
    # torch.save(model_int8,INT8_MODEL_PATH)
    
    model_int8.eval()
    scripted = torch.jit.script(model_int8)
    scripted.save("")

    print("FX static quantization completed successfully")
    print(f"Saved to: {INT8_MODEL_PATH}")
    
    print(model_int8)


if __name__ == "__main__":
    main()


Loading FP32 model...
FP32 MOdel  ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 25

/home/intellisense08/anaconda3/lib/python3.11/site-packages/torch/ao/quantization/observer.py:244: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


Loading calibration data...
Running calibration...
Converting to INT8...
After converting int8 model  GraphModule(
  (conv1): QuantizedConvReLU2d(3, 64, kernel_size=(7, 7), stride=(2, 2), scale=0.05465181544423103, zero_point=0, padding=(3, 3))
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Module(
    (0): Module(
      (conv1): QuantizedConvReLU2d(64, 64, kernel_size=(1, 1), stride=(1, 1), scale=0.06759272515773773, zero_point=0)
      (conv2): QuantizedConvReLU2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.06302446871995926, zero_point=0, padding=(1, 1))
      (conv3): QuantizedConv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), scale=0.1327408254146576, zero_point=63)
      (downsample): Module(
        (0): QuantizedConv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), scale=0.11317262053489685, zero_point=64)
      )
    )
    (1): Module(
      (conv1): QuantizedConvReLU2d(256, 64, kernel_size=(1, 1), stride=(1, 1), scale=0

/home/intellisense08/anaconda3/lib/python3.11/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(


FX static quantization completed successfully
Saved to: /home/intellisense08/Yehan/project_dir/Real_World_data/quantization/models/round2/resnet50_syn_int8_2.pth
GraphModule(
  (conv1): QuantizedConvReLU2d(3, 64, kernel_size=(7, 7), stride=(2, 2), scale=0.05465181544423103, zero_point=0, padding=(3, 3))
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Module(
    (0): Module(
      (conv1): QuantizedConvReLU2d(64, 64, kernel_size=(1, 1), stride=(1, 1), scale=0.06759272515773773, zero_point=0)
      (conv2): QuantizedConvReLU2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.06302446871995926, zero_point=0, padding=(1, 1))
      (conv3): QuantizedConv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), scale=0.1327408254146576, zero_point=63)
      (downsample): Module(
        (0): QuantizedConv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), scale=0.11317262053489685, zero_point=64)
      )
    )
    (1): Module(
      (conv1): QuantizedCo